# Лабораторная 09. Cache и память

Цель: понять, когда cache помогает, а когда просто занимает memory.

In [1]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName('lab-09-cache').master('local[*]')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.adaptive.enabled', 'false')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
base_uri = Path('spark_core_data').absolute().as_uri()
orders = spark.read.parquet(f'{base_uri}/orders')
print('Spark UI:', spark.sparkContext.uiWebUrl)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/08 14:34:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

Spark UI: http://c6cfc051eacd:4040


## Без cache
Один и тот же filtered DataFrame используется несколько раз. Spark может пересчитывать lineage для каждой action.

In [2]:
paid = orders.filter(F.col('status') == 'paid').select('order_id', 'customer_id', 'order_amount')
paid.count()
paid.groupBy('customer_id').count().count()

2500

## С cache
`cache()` только помечает DataFrame. Чтобы реально положить данные в cache, нужна action.

In [3]:
paid_cached = paid.cache()

In [4]:
paid_cached.count()  # materialization
print('Откройте Storage tab:', spark.sparkContext.uiWebUrl)

[Stage 6:>                                                          (0 + 4) / 4]

Откройте Storage tab: http://c6cfc051eacd:4040


In [5]:
paid_cached.groupBy('customer_id').count().count()
paid_cached.agg(F.sum('order_amount')).show()

+-----------------+
|sum(order_amount)|
+-----------------+
|       8117202.62|
+-----------------+



## Очистка cache

In [6]:
paid_cached.unpersist()

DataFrame[order_id: bigint, customer_id: bigint, order_amount: decimal(10,2)]

Вопросы:

- Почему после `cache()` нужно вызвать action? чтобы даннные записались в кэш
- Почему cache помогает только при повторном использовании? потому что в первый раз в любом случае придется выполнить вычисления с нуля
- Что видно на Storage tab? после action появлилсь данные о кэш - RDD name, storage level, Cached Partitions, Fraction Cached, Size in Memory, Size on Disk
- Что будет, если закэшировать слишком большой DataFrame? fraction cached < 100.00%
- Когда cache лучше не использовать? когда данные используются один раз или когда их слишком много и они не помещаются в памяти

In [7]:
spark.stop()